# 03 - Modelo baseline CNN

CNN construida desde cero para tener un punto de partida. No espero resultados muy buenos aca, la idea es ver cuanto mejora el transfer learning en comparacion. Uso 64x64 en lugar de 224x224 para poder correrlo rapido en Colab, con imagenes grandes y sin pesos preentrenados el entrenamiento dura horas sin necesidad.

In [ ]:
!pip install tensorflow-datasets gdown scikit-learn seaborn --quiet

In [ ]:
import os

WORK_PATH = '/content/plantvillage'
os.makedirs(WORK_PATH, exist_ok=True)

In [ ]:
import gdown
gdown.download_folder(
    'https://drive.google.com/drive/folders/1OCOyDSR9C3TCzLthsMxJCmy6wcjM3pxs',
    output=WORK_PATH,
    quiet=True
)

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.metrics import classification_report, confusion_matrix

# precision mixta para aprovechar los tensor cores de la GPU
tf.keras.mixed_precision.set_global_policy('mixed_float16')

print('tensorflow:', tf.__version__)
print('gpu disponible:', tf.config.list_physical_devices('GPU'))
print('precision global:', tf.keras.mixed_precision.global_policy().name)

tf.random.set_seed(42)
np.random.seed(42)

## Parametros y datos

In [ ]:
with open(f'{WORK_PATH}/dataset_info.json') as f:
    ds_info = json.load(f)
with open(f'{WORK_PATH}/class_weight.json') as f:
    class_weight = {int(k): v for k, v in json.load(f).items()}

NUM_CLASSES = ds_info['num_classes']
CLASS_NAMES = ds_info['class_names']

# 64x64 es suficiente para una CNN desde cero y reduce el tiempo de entreno drasticamente
IMG_SIZE   = 64
BATCH_SIZE = 64
AUTOTUNE   = tf.data.AUTOTUNE

print('clases:', NUM_CLASSES)
print('tamano imagen:', IMG_SIZE)
print('batch size:', BATCH_SIZE)

In [ ]:
(ds_train_raw, ds_val_raw, ds_test_raw), _ = tfds.load(
    'plant_village',
    split=['train[:70%]', 'train[70%:85%]', 'train[85%:]'],
    with_info=True,
    as_supervised=True,
    shuffle_files=True,
    read_config=tfds.ReadConfig(shuffle_seed=42)
)

# solo redimensiona; el augmentation va dentro del modelo
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    return image, label

ds_train = (
    ds_train_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .shuffle(2000, seed=42)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
ds_val = (
    ds_val_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
ds_test = (
    ds_test_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print('pipeline listo')

## Arquitectura CNN baseline

Cuatro bloques Conv2D + BatchNormalization + ReLU + MaxPooling. Puse el augmentation dentro del modelo para que se desactive automaticamente cuando evaluo.

In [ ]:
def build_baseline_cnn(num_classes, img_size=64):
    inputs = tf.keras.Input(shape=(img_size, img_size, 3))

    # augmentation dentro del modelo: activo en entrenamiento, inactivo al evaluar
    x = tf.keras.layers.RandomFlip('horizontal_and_vertical')(inputs)
    x = tf.keras.layers.RandomRotation(0.2)(x)
    x = tf.keras.layers.RandomZoom(0.2)(x)
    x = tf.keras.layers.RandomBrightness(0.2, value_range=(0, 1))(x)
    x = tf.keras.layers.RandomContrast(0.2)(x)

    for filters in [32, 64, 128, 256]:
        x = tf.keras.layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation('relu')(x)
        x = tf.keras.layers.MaxPooling2D(2)(x)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    # float32 en la salida para evitar problemas de precision
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax', dtype='float32')(x)

    return tf.keras.Model(inputs, outputs, name='cnn_baseline')

model = build_baseline_cnn(NUM_CLASSES)
model.summary()

## Entrenamiento

Uso sparse_categorical_crossentropy porque los labels son enteros. El class_weight ayuda a que el modelo no ignore las clases con pocas imagenes.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=f'{WORK_PATH}/baseline_best.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, verbose=1
    )
]

history = model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=30,
    class_weight=class_weight,
    callbacks=callbacks
)

## Curvas de aprendizaje

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'],     label='train')
ax1.plot(history.history['val_accuracy'], label='val')
ax1.set_title('accuracy')
ax1.set_xlabel('epoca')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(history.history['loss'],     label='train')
ax2.plot(history.history['val_loss'], label='val')
ax2.set_title('loss')
ax2.set_xlabel('epoca')
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle('CNN baseline — curvas de aprendizaje', fontsize=12)
plt.tight_layout()
plt.savefig(f'{WORK_PATH}/baseline_curvas.png', dpi=150)
plt.show()

## Evaluacion en test

Evaluo con el mejor modelo que guardo el callback, no con el del ultimo epoch.

In [ ]:
best_model = tf.keras.models.load_model(f'{WORK_PATH}/baseline_best.keras')

# recorro el dataset una sola vez para que pred y labels esten alineados
y_pred_list, y_true_list = [], []
for images, labels in ds_test:
    preds = best_model(images, training=False)
    y_pred_list.extend(np.argmax(preds.numpy(), axis=1))
    y_true_list.extend(labels.numpy())

y_pred = np.array(y_pred_list)
y_true = np.array(y_true_list)

test_acc = np.mean(y_pred == y_true)
print(f'accuracy en test: {test_acc:.4f} ({test_acc*100:.2f}%)')
print()
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred, normalize='true')

plt.figure(figsize=(22, 20))
sns.heatmap(cm, annot=False, cmap='Greens',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('matriz de confusion normalizada — CNN baseline', fontsize=13)
plt.xlabel('prediccion')
plt.ylabel('real')
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()
plt.savefig(f'{WORK_PATH}/baseline_confusion.png', dpi=150)
plt.show()

In [ ]:
resultados_baseline = {
    'modelo'   : 'CNN baseline',
    'img_size' : IMG_SIZE,
    'test_acc' : float(test_acc),
    'epochs'   : len(history.history['loss'])
}

with open(f'{WORK_PATH}/resultados_baseline.json', 'w') as f:
    json.dump(resultados_baseline, f, indent=2)

print('resultados guardados en:', WORK_PATH)